# Week 07 — Neural Network Surrogate Training

Same architecture family as W4-W6: MLP with H ∈ {16, 32}, four regularisation variants (plain / dropout / weight-decay / ensemble), 5-fold CV across 8 configs.

Re-trains on the latest data including W6 portal returns (16/16/21/36/26/26/36/46 pts).

F2/F4/F5/F6/F8 had new bests in W6 — fresh NN gradients at these new best points.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_07')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 16 pts, 2D, baseline RMSE = 0.0018
  All configs (sorted):
    H= 16  wd          CV RMSE = 0.0021 ← BEST
    H= 16  plain       CV RMSE = 0.0022
    H= 32  dropout     CV RMSE = 0.0024
    H= 32  ensemble    CV RMSE = 0.0026
    H= 16  dropout     CV RMSE = 0.0028
    H= 16  ensemble    CV RMSE = 0.0028
    H= 32  plain       CV RMSE = 0.0030
    H= 32  wd          CV RMSE = 0.0031
  → best: H=16, wd, RMSE=0.0021 (-17.9% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.05700000002980232, -0.03200000151991844]



F2: 16 pts, 2D, baseline RMSE = 0.2433
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.2023 ← BEST
    H= 32  dropout     CV RMSE = 0.2106
    H= 16  wd          CV RMSE = 0.2710
    H= 32  ensemble    CV RMSE = 0.2720
    H= 16  plain       CV RMSE = 0.2743
    H= 16  ensemble    CV RMSE = 0.2884
    H= 32  wd          CV RMSE = 0.2990
    H= 32  plain       CV RMSE = 0.3072
  → best: H=16, dropout, RMSE=0.2023 (+16.9% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.5860000252723694, 1.3339999914169312]



F3: 21 pts, 3D, baseline RMSE = 0.0752
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.0696 ← BEST
    H= 32  wd          CV RMSE = 0.0733
    H= 32  ensemble    CV RMSE = 0.0751
    H= 16  wd          CV RMSE = 0.0772
    H= 32  plain       CV RMSE = 0.0791
    H= 16  plain       CV RMSE = 0.0797
    H= 32  dropout     CV RMSE = 0.0917
    H= 16  dropout     CV RMSE = 0.0939
  → best: H=16, ensemble, RMSE=0.0696 (+7.5% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.013000000268220901, -0.010999999940395355, 0.23399999737739563]



F4: 36 pts, 4D, baseline RMSE = 9.0968
  All configs (sorted):
    H= 32  wd          CV RMSE = 4.3461 ← BEST
    H= 32  plain       CV RMSE = 4.4030
    H= 32  ensemble    CV RMSE = 4.7082
    H= 16  ensemble    CV RMSE = 4.9081
    H= 32  dropout     CV RMSE = 5.2922
    H= 16  plain       CV RMSE = 5.4865
    H= 16  wd          CV RMSE = 5.4883
    H= 16  dropout     CV RMSE = 5.5361
  → best: H=32, wd, RMSE=4.3461 (+52.2% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-13.717000007629395, 12.083999633789062, -4.565999984741211, 3.296999931335449]



F5: 26 pts, 4D, baseline RMSE = 768.4140
  All configs (sorted):
    H= 32  plain       CV RMSE = 72.8596 ← BEST
    H= 32  ensemble    CV RMSE = 73.8989
    H= 32  wd          CV RMSE = 76.6073
    H= 16  plain       CV RMSE = 84.0157
    H= 16  ensemble    CV RMSE = 85.1327
    H= 16  wd          CV RMSE = 86.8159
    H= 32  dropout     CV RMSE = 196.3125
    H= 16  dropout     CV RMSE = 221.2281
  → best: H=32, plain, RMSE=72.8596 (+90.5% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [5297.7060546875, 3519.89208984375, 6299.84912109375, 6400.240234375]



F6: 26 pts, 5D, baseline RMSE = 0.6309
  All configs (sorted):
    H= 32  wd          CV RMSE = 0.3603 ← BEST
    H= 32  ensemble    CV RMSE = 0.3622
    H= 32  plain       CV RMSE = 0.3627
    H= 16  ensemble    CV RMSE = 0.3633
    H= 16  plain       CV RMSE = 0.3789
    H= 16  wd          CV RMSE = 0.3789
    H= 32  dropout     CV RMSE = 0.4174
    H= 16  dropout     CV RMSE = 0.4343
  → best: H=32, wd, RMSE=0.3603 (+42.9% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.5910000205039978, -0.8560000061988831, -0.14499999582767487, -0.24300000071525574, -1.3240000009536743]



F7: 36 pts, 6D, baseline RMSE = 0.5091
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.3280 ← BEST
    H= 32  dropout     CV RMSE = 0.3428
    H= 32  ensemble    CV RMSE = 0.4713
    H= 32  wd          CV RMSE = 0.4812
    H= 32  plain       CV RMSE = 0.4965
    H= 16  ensemble    CV RMSE = 0.5067
    H= 16  wd          CV RMSE = 0.5582
    H= 16  plain       CV RMSE = 0.5861
  → best: H=16, dropout, RMSE=0.3280 (+35.6% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.3199999928474426, -0.009999999776482582, -0.10000000149011612, -0.1940000057220459, -0.21699999272823334, 0.2770000100135803]



F8: 46 pts, 8D, baseline RMSE = 1.1185
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.3616 ← BEST
    H= 32  plain       CV RMSE = 0.3675
    H= 32  wd          CV RMSE = 0.3707
    H= 16  ensemble    CV RMSE = 0.3742
    H= 16  dropout     CV RMSE = 0.4001
    H= 32  dropout     CV RMSE = 0.4139
    H= 16  wd          CV RMSE = 0.4536
    H= 16  plain       CV RMSE = 0.4630
  → best: H=32, ensemble, RMSE=0.3616 (+67.7% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.5220000147819519, 0.05700000002980232, 0.22200000286102295, 0.1860000044107437, 0.5329999923706055, 0.06199999898672104, -0.3880000114440918, 0.057999998331069946]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 16 pts, 11 positive, 5 zero-or-negative (69% positive)


Sign classifier trained. LOO accuracy = 68.75%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  wd              0.0021      0.0018    -17.9%       ✗
 2   16  dropout         0.2023      0.2433    +16.9%       ✓
 3   16  ensemble        0.0696      0.0752     +7.5%       ✓
 4   32  wd              4.3461      9.0968    +52.2%       ✓
 5   32  plain          72.8596    768.4140    +90.5%       ✓
 6   32  wd              0.3603      0.6309    +42.9%       ✓
 7   16  dropout         0.3280      0.5091    +35.6%       ✓
 8   32  ensemble        0.3616      1.1185    +67.7%       ✓
